In [ ]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph


class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percent: float
    summary: str


def calculate_sr(state: BatsmanState) -> dict:
    balls = state["balls"]
    runs = state["runs"]
    sr = (runs / balls * 100) if balls > 0 else 0.0
    return {"sr": round(sr, 2)}


def calculate_bpb(state: BatsmanState) -> dict:
    balls = state["balls"]
    boundaries = state["fours"] + state["sixes"]
    bpb = (balls / boundaries) if boundaries > 0 else 0.0
    return {"bpb": round(bpb, 2)}


def calculate_boundary_percent(state: BatsmanState) -> dict:
    runs = state["runs"]
    boundary_runs = (state["fours"] * 4) + (state["sixes"] * 6)
    boundary_percent = (boundary_runs / runs * 100) if runs > 0 else 0.0
    return {"boundary_percent": round(boundary_percent, 2)}


def summary(state: BatsmanState) -> dict:
    summary_text = (
        f"Strike Rate: {state['sr']}\n"
        f"Balls per Boundary: {state['bpb']}\n"
        f"Boundary Percent: {state['boundary_percent']}%"
    )
    return {"summary": summary_text}


graph = StateGraph(BatsmanState)

graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_boundary_percent", calculate_boundary_percent)
graph.add_node("summary", summary)

graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge(START, "calculate_boundary_percent")

graph.add_edge("calculate_sr", "summary")
graph.add_edge("calculate_bpb", "summary")
graph.add_edge("calculate_boundary_percent", "summary")

graph.add_edge("summary", END)

workflow = graph.compile()

initial_state = {"runs": 100, "balls": 50, "fours": 6, "sixes": 4}

final_state = workflow.invoke(initial_state)

print(final_state["summary"])

Strike Rate: 200.0
Balls per Boundary: 5.0
Boundary Percent: 48.0%
